In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    is_premium: bool = False

In [3]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    user_name = request.runtime.context.user_name
    is_premium = request.runtime.context.is_premium

    if is_premium:
        model_name = "google_genai:gemini-3.5-flash"
        print(f"{user_name} 사용자는 프리미엄 회원입니다. gemini-3.5-flash를 호출합니다.")
    else:
        model_name = "google_genai:gemini-2.5-flash"
        print(f"{user_name} 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)  


In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "너는 누구고, 모델 코드가 뭐야?"}]},
    context=Context(user_name="김일남", is_premium=True)
)

In [9]:
agent.invoke(
    {"messages": [{"role": "user", "content": "너는 누구고, 모델 코드가 뭐야?"}]},
    context=Context(user_name="김일남", is_premium=False)
)

김일남 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.


{'messages': [HumanMessage(content='너는 누구고, 모델 코드가 뭐야?', additional_kwargs={}, response_metadata={}, id='c7c6d406-50f4-4078-8e60-1b304edb1cce'),
  AIMessage(content='저는 Google에서 훈련한 대규모 언어 모델입니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7cb8-61b9-7593-af60-9e439d24916f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 131, 'total_tokens': 142, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 118}})]}

---

In [11]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    print("msg_len:", msg_len)

    if msg_len < 10:
        model_name = "google_genai:gemini-2.5-flash-lite"
        print("사용 모델: gemini-2.5-flash-lite")
    elif msg_len < 30:
        model_name = "google_genai:gemini-3.1-flash-lite"
        print("사용 모델: gemini-3.1-flash-lite")
    else:
        model_name = "google_genai:gemini-3.5-flash"
        print("사용 모델: gemini-3.5-flash")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)


In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[dynamic_model_selector],
)

In [14]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
)

msg_len: 9
사용 모델: gemini-2.5-flash-lite


{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='a0bfa331-cc96-463e-8715-173d39f77339'),
  AIMessage(content='저는 대규모 언어 모델이며, 이름이 없습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7cbf-c65b-7890-84f9-5c0d6201346c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 12, 'total_tokens': 18, 'input_token_details': {'cache_read': 0}})]}

In [15]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김일남야. 아이스 아메리카노를 좋아해."}]},
)

msg_len: 27
사용 모델: gemini-3.1-flash-lite


{'messages': [HumanMessage(content='내 이름은 김일남야. 아이스 아메리카노를 좋아해.', additional_kwargs={}, response_metadata={}, id='10fbf238-eff2-44b5-b3a4-9df265084ce6'),
  AIMessage(content=[{'type': 'text', 'text': "반가워요, 일남 님! \n\n아이스 아메리카노 좋아하시는군요. 역시 '얼죽아(얼어 죽어도 아이스 아메리카노)'인가요? 오늘 날씨에 딱 시원하게 한 잔 마시면 정말 최고죠!\n\n어떤 스타일의 아메리카노를 제일 좋아하세요? 고소한 원두를 좋아하시나요, 아니면 산미가 있는 걸 좋아하시나요? \n\n일남 님과 대화하게 되어 기뻐요! 또 궁금한 점이나 나누고 싶은 이야기가 있으면 언제든 말씀해 주세요. :)", 'extras': {'signature': 'EjQKMgEMOdbHcRnCGTvz/uAH0qzwANUcjxPNoL8RewMXGYrSSdOP4UMD0S9yEisgoLuKWjHh'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7cc0-763a-7de2-ba9e-03c5a0b97e2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 126, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}})]}

In [16]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김일남야. 아이스 아메리카노를 좋아해. 랭체인을 공부하고 있고, 사이드 프로젝트로 AI 애플리케이션을 개발하고 있어. 내가 알아야할 내용이 뭘까?"}]},
)

msg_len: 87
사용 모델: gemini-3.5-flash


{'messages': [HumanMessage(content='내 이름은 김일남야. 아이스 아메리카노를 좋아해. 랭체인을 공부하고 있고, 사이드 프로젝트로 AI 애플리케이션을 개발하고 있어. 내가 알아야할 내용이 뭘까?', additional_kwargs={}, response_metadata={}, id='88d6aa95-0f5c-4f11-8fd2-aa0cbe775f18'),
  AIMessage(content=[{'type': 'text', 'text': '반갑습니다, 일남 님! 개발자의 생명수이자 소울 드링크인 **시원한 아이스 아메리카노(아아)** 한 잔 옆에 두고 열공 중이시겠군요. \n\n랭체인(LangChain)을 학습하면서 동시에 AI 사이드 프로젝트를 진행하고 계시다니 아주 훌륭한 접근입니다. 이론만 공부하는 것보다 직접 만들면서 배울 때 성장 속도가 가장 빠르니까요.\n\n현재 시점에서 일남 님이 **기술적으로, 그리고 프로젝트 성공을 위해 꼭 알아야 할 핵심 내용**들을 정리해 드립니다. 아아 한 모금 마시면서 가볍게 읽어보세요!\n\n---\n\n### 1. 랭체인(LangChain) 공부할 때 꼭 짚고 넘어가야 할 3가지\n\n요즘 랭체인은 업데이트가 매우 빠릅니다. 옛날 튜토리얼을 보면 코드가 작동하지 않는 경우가 많으니, 아래 개념을 중심으로 공식 문서를 보시는 것을 추천합니다.\n\n*   **LCEL (LangChain Expression Language):** \n    *   랭체인의 핵심 중의 핵심입니다. 이전의 복잡한 Chain 구조 대신 `|` (파이프) 연산자를 사용해 체인을 직관적으로 연결하는 방식입니다. 이걸 모르면 최신 랭체인 코드를 이해하기 어렵습니다.\n*   **RAG (Retrieval-Augmented Generation, 검색 증강 생성):**\n    *   AI 앱의 80%는 "내 데이터를 기반으로 답변하는 챗봇"입니다. 문서를 쪼개고(Split), 벡터로 변환하여(Embedding), 벡터 데이터